# Tool Calling 完整流程

本 Notebook 演示如何：

1. 定义带 JSON Schema 的 `Tool`
2. 在流式请求中让模型生成 `ToolCall`
3. 本地执行工具函数
4. 构造 `ToolResultMessage` 并继续请求获取最终回答

运行前请设置 `VOLCENGINE_API_KEY` 或其他可用厂商 Key。


In [ ]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"


## 步骤 1：定义工具并首次请求

把 `Tool` 对象放入 `Context.tools`，模型会在需要时生成 `toolCall` 内容块。


In [ ]:
import json
from nova_ai import (
    Tool, get_model, UserMessage, Context, stream_simple,
    ToolResultMessage, TextContent,
)

weather_tool = Tool(
    name="get_weather",
    description="查询指定城市的当前天气",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "城市名称，如 北京"},
        },
        "required": ["city"],
    },
)

def get_weather(city: str) -> str:
    # 这里应该是真实 API 调用，示例中直接返回模拟数据
    return f"{city} 今天多云，气温 22-28°C。"

model = get_model("volcengine", "deepseek-v3-2-251201")

context = Context(
    system_prompt="你可以调用 get_weather 工具查询天气。",
    messages=[UserMessage(role="user", content="杭州今天天气怎么样？")],
    tools=[weather_tool],
)

stream = stream_simple(model, context)
async for event in stream:
    if event.type == "text_delta":
        print(event.delta, end="", flush=True)
    elif event.type == "toolcall_start":
        print(f"\n[工具调用开始]")
    elif event.type == "toolcall_delta":
        print(event.delta, end="", flush=True)
    elif event.type == "done":
        print("\n[助手消息完成]")

assistant_msg = await stream.result()
print("\n助手消息内容块:", assistant_msg.content)


## 步骤 2：执行工具并继续对话

从 assistant 消息中取出 `ToolCall`，执行后将 `ToolResultMessage` 追加到上下文，再调一次模型即可得到最终回答。


In [ ]:
# 必须先把 assistant 消息加入上下文，再追加 toolResult
context.messages.append(assistant_msg)

tool_calls = [c for c in assistant_msg.content if c.type == "toolCall"]
print("待执行 tool calls:", tool_calls)

for tc in tool_calls:
    if tc.name == "get_weather":
        result_text = get_weather(tc.arguments.get("city", "未知城市"))
        context.messages.append(
            ToolResultMessage(
                role="toolResult",
                tool_call_id=tc.id,
                tool_name=tc.name,
                content=[TextContent(text=result_text)],
            )
        )

# 再次调用模型，把工具结果交给它总结
stream2 = stream_simple(model, context)
print("\n最终回答:")
async for event in stream2:
    if event.type == "text_delta":
        print(event.delta, end="", flush=True)

final = await stream2.result()
print("\n停止原因:", final.stop_reason)
